# Data ETL and exploration/preprocessing

## Loading and Exploring Data

### Guidelines

1. Load all the kcmetro text data independently and drop irrelevant information
2. Merge all data 
3. Explore data and calendar/caldates/service id in particular
4. Create function to calculate frequencies from schedule data
5. create function to map routes that match frequency target 

Libraries to Use:

-numpy and pandas 
-geopandas and folium
-datetime 
-os and shutils 
-arcpy

In [64]:
###installs

%pip install folium pandas numpy geopandas

Note: you may need to restart the kernel to use updated packages.


In [65]:
#imports
import numpy as np, pandas as pd
import folium as fl, geopandas as gp 
from datetime import datetime


In [66]:
###loading data
trips = pd.read_csv(r'../data/KCM_data/trips.txt')
stop_times = pd.read_csv(r'../data/KCM_data/stop_times.txt')
stops = pd.read_csv(r'../data/KCM_data/stops.txt')
routes = pd.read_csv(r'../data/KCM_data/routes.txt')
cal = pd.read_csv(r'../data/KCM_data/calendar.txt')
cal_dates = pd.read_csv(r'../data/KCM_data/calendar_dates.txt')

In [67]:
###dropping unnecessary information

trips_clean = trips[['route_id','service_id','trip_id','trip_headsign', 'direction_id','block_id','shape_id']]
stop_times_clean = stop_times[['trip_id','arrival_time','departure_time','stop_id','stop_sequence','shape_dist_traveled','timepoint']]
stops_clean = stops[['stop_id','stop_code','stop_name','stop_lat','stop_lon']]
routes_clean = routes[['route_id','agency_id','route_short_name','route_desc','route_type','route_color','route_text_color']]

###cal and calendar dates are to remain unchanged until I explore the data more thoroughly 
###set the variables to the clean dataframes

trips = trips_clean
stop_times = stop_times_clean
stops = stops_clean
routes = routes_clean


###find and set types
trips.service_id = trips.service_id.astype(int)
stop_times.arrival_time = pd.to_timedelta(stop_times.arrival_time)
stop_times.departure_time = pd.to_timedelta(stop_times.departure_time)



In [68]:
###merge together data

#merge all files one by one 

data_merged = pd.merge(trips, stop_times, how='left', on='trip_id')
data_merged = pd.merge(data_merged, stops, how='left', on='stop_id')
data_merged = pd.merge(data_merged, routes, how='left', on='route_id')

###filter to only kcm bus routes 
data_merged = data_merged.loc[data_merged.agency_id == 1].drop('agency_id', axis=1)
data_merged.head(5)

,route_id,service_id,trip_id,trip_headsign,direction_id,block_id,shape_id,arrival_time,departure_time,stop_id,...,timepoint,stop_code,stop_name,stop_lat,stop_lon,route_short_name,route_desc,route_type,route_color,route_text_color
0,100001,151056,562518401,Kinnear Seattle Center W,0,8222244,11001004,0 days 05:02:00,0 days 05:02:00,1530,...,1,1530,S Jackson St & 5th Ave S,47.599266,-122.328384,1,Kinnear - Downtown Seattle,3,FDB71A,000000
1,100001,151056,562518401,Kinnear Seattle Center W,0,8222244,11001004,0 days 05:03:43,0 days 05:03:43,1610,...,0,1610,Prefontaine Pl S & Yesler Way,47.601658,-122.329941,1,Kinnear - Downtown Seattle,3,FDB71A,000000
2,100001,151056,562518401,Kinnear Seattle Center W,0,8222244,11001004,0 days 05:05:24,0 days 05:05:24,538,...,0,538,3rd Ave & Columbia St,47.604095,-122.332504,1,Kinnear - Downtown Seattle,3,FDB71A,000000
3,100001,151056,562518401,Kinnear Seattle Center W,0,8222244,11001004,0 days 05:07:19,0 days 05:07:19,558,...,0,558,3rd Ave & Seneca St,47.607052,-122.335213,1,Kinnear - Downtown Seattle,3,FDB71A,000000
4,100001,151056,562518401,Kinnear Seattle Center W,0,8222244,11001004,0 days 05:09:00,0 days 05:09:00,575,...,1,575,3rd Ave & Pike St,47.609642,-122.337585,1,Kinnear - Downtown Seattle,3,FDB71A,000000


### Data Exploration

In [69]:
###let us explore the service id to see which routes have which ids

route_servs = pd.DataFrame(data_merged.groupby('route_short_name').service_id.unique())
route_servs['unique_service_ids'] = route_servs.service_id.apply(lambda x: x.shape[0])

###put unique service ids in the front of the dataframe 
route_servs = route_servs.iloc[:,[1,0]]

print(route_servs.head(6))

                  unique_service_ids              service_id
route_short_name                                            
1                                  3  [151056, 30098, 34059]
10                                 3  [151056, 30098, 34059]
101                                3  [151056, 30098, 34059]
102                                1                [151056]
105                                3  [151056, 30098, 34059]
106                                3  [151056, 30098, 34059]


As we can see, the most popular service ids are 151056, 30098, and 34059 - corresponding to weekday,sat, and sun service

In [88]:
####find calendar dates to confirm my intuition 
cal_serv = cal

###add total weekday count and set to one for boolean comparisons 
cal_serv['weekday'] = cal_serv.drop(columns=['service_id', 'start_date','end_date', 'saturday','sunday']).apply(lambda x: any(x), axis=1)

####drop individual day of the workweek columns 
cal_serv = cal_serv.drop(columns=cal_serv.columns[1:6])

### us DF.dot to multiply 
cal_serv['dow'] = cal_serv[['saturday','sunday','weekday']].dot(cal_serv.columns[[1,2,5]]).str.lower()

###find services that are weekday, weekend, or special service
cal_serv = cal_serv.drop(columns=['saturday','sunday','weekday'])
cal_serv.dow = cal_serv.dow.replace('', 'special')

###make service id an int
cal_serv.service_id = cal_serv.service_id.astype(int)

###save service ids by dow for further exploration
cal_serv.to_csv('../data/serv_dow.csv')

###display 
cal_serv.head(8)


,service_id,start_date,end_date,dow
0,103294,20260902,20270326,special
1,125810,20260902,20270326,special
2,128191,20260902,20270326,special
3,1420,20260907,20260907,weekday
4,145779,20260831,20270326,weekday
5,147726,20260831,20270326,weekday
6,149182,20260831,20270326,weekday
7,151056,20260831,20270326,weekday


### Frequency Calculations/Functions and Verification

We will explore the data by using various methods to try to calculate frequency and then we will validate. 

In [ ]:
##lets use groupby in pandas to filter/sort the data we desire and limit it to a single stop + dir + service_id 
data_merged = data_merged.sort_values(by=['stop_id', 'arrival_time'])

####group by stop id and sort by time of arrival 
data_merged['prev_depart'] = data_merged.groupby(['route_id', 'service_id', 'direction_id', 'stop_id']).departure_time.shift(1)
data_merged['gap'] = data_merged['departure_time'] - data_merged['prev_depart']
route_freq = pd.DataFrame(data_merged.groupby('route_short_name').gap.mean().sort_values(ascending=True))

##group by stop sequence and sort by time of arrival
data_merged['prev_depart'] = data_merged.groupby(['route_id', 'service_id', 'direction_id', 'stop_sequence']).departure_time.shift(1)
data_merged['gap'] = data_merged['departure_time'] - data_merged['prev_depart']
route_freq['by_sequence'] = pd.DataFrame(data_merged.groupby('route_short_name').gap.mean().sort_values(ascending=True))


### we will now find the differences between sorting by stop sequence and sorting by stop_id 
route_freq['dif'] = np.abs(route_freq['by_sequence'] - route_freq['gap'])

###display the differences based on calculation type. 
###it would seem sensible to me that stop id would be the better variable to use as lines can start at various stop ids first (changing stop seq no.)
route_freq.loc[route_freq.dif >= pd.Timedelta(seconds=1)].sort_values('dif')['dif']

route_short_name
107                              0 days 00:00:07.554677
161                              0 days 00:00:08.477619
A Line                           0 days 00:00:09.365427
E Line                           0 days 00:00:10.490623
245                              0 days 00:00:15.483792
D Line                           0 days 00:00:16.952300
215                              0 days 00:00:19.084447
Trailhead Direct Issaquah Alps   0 days 00:00:21.612699
226                              0 days 00:00:25.181473
164                              0 days 00:00:25.711766
125                              0 days 00:00:27.755989
27                               0 days 00:00:30.325922
914                              0 days 00:00:30.377925
49                               0 days 00:00:30.985153
223                              0 days 00:00:34.156468
62                               0 days 00:00:36.991937
106                              0 days 00:00:54.604609
204                            

From here we will validate data using a few routes and accurate schedule data

In [99]:
###find data from rapidrides to compare


rr_data = data_merged.loc[data_merged.route_short_name.str.match(r'\w Line')]
rr_freq = route_freq.loc[route_freq.index.str.match(r'\w Line')]


In [141]:
###function for gapping !!! FINISH !!! and test

def freq_calc(df, label='departure_time', start_hr=0, end_hr=30):

    ### only locate columns we need for the calculations to save memory and increase run speed
    df_lite = df[['route_id','trip_id','stop_id','route_short_name', 'service_id','direction_id', label]]

    start_time = pd.Timedelta(hours=start_hr)
    end_time = pd.Timedelta(hours=end_hr)
    ###convert departure_time to timedelta
    df_lite.departure_time = pd.to_timedelta(df_lite.departure_time)
    ###restrict to time range
    df_lite = df_lite.loc[(df_lite.departure_time >= start_time) & (df_lite.departure_time <= end_time)]

    ####group by stop id and sort by time of arrival 
    df_lite['prev_depart'] = df_lite.groupby(['route_id', 'service_id', 'direction_id', 'stop_id']).departure_time.shift(1)
    df_lite['gap'] = df_lite['departure_time'] - df_lite['prev_depart']

    print(df_lite.departure_time.head())
    ###calculate frequencies 
    route_freq = pd.DataFrame(df_lite.groupby('route_short_name').gap.mean().sort_values(ascending=True))

    return route_freq


    

In [142]:
freq_calc(rr_data)

1008287   0 days 04:30:00
1012193   0 days 04:30:00
1017044   0 days 04:30:00
1008938   0 days 05:00:00
1012130   0 days 05:00:00
Name: departure_time, dtype: timedelta64[us]


,gap
route_short_name,
G Line,0 days 00:10:08.729701
E Line,0 days 00:12:15.598914
A Line,0 days 00:13:50.505544
B Line,0 days 00:15:01.652895
H Line,0 days 00:15:05.484830
D Line,0 days 00:15:23.089778
C Line,0 days 00:15:35.929081
F Line,0 days 00:15:56.551872


### Displaying Data for User